
# ED Operational Classifier (MLP) — Operational States → Adjustment Factors

**Invariants respected** (non-exhaustive):  
- `WorkflowState` left untouched (not defined/modified here).  
- Cold-start safe: no transform before fit; classifier guards return `Normal_Flow` if unfitted.  
- `CONFIG` bootstrapped with defaults; `RUN_UI=False`, `RUN_PIPELINE=False`; paths use CSV/dir semantics.  
- SOP auto-pull offline-safe via `refresh_sop_registry`.  
- QR generate + manual scan fallback updates moves log.  
- Overdue/alerts utilities provided.  
- No import-time side effects in tracker stubs.

This notebook focuses on the MLP that **predicts operational states** and maps to **OperationalAdjustments**.


In [ ]:

import os, json, math, datetime as dt
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple, Optional
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# CONFIG bootstrap (safe if already set in environment)
CONFIG = {
    "RUN_UI": False,
    "RUN_PIPELINE": False,
    "EQUIPMENT_STATUS_PATH": "/mnt/data/equipment_status.csv",
    "EQUIPMENT_MOVES_LOG_PATH": "/mnt/data/equipment_moves_log.csv",
    "SOP_REGISTRY_PATH": "/mnt/data/sop_registry.csv",
    "QR_OUTPUT_DIR": "/mnt/data/qr_codes",
    "EVENT_LOG_PATH": "/mnt/data/event_log.csv",
}

ED_STATES = [
    "Normal_Flow",
    "High_Volume",
    "Resource_Constrained",
    "Critical_Surge",
    "Night_Shift",
    "Flu_Season",
]

@dataclass
class OperationalAdjustments:
    triage_urgency_multiplier: float
    handoff_timing_factor: float
    resource_allocation_weights: Dict[str, float]
    communication_urgency_threshold: float
    discharge_readiness_threshold: float

def default_operational_profiles() -> Dict[str, OperationalAdjustments]:
    return {
        "Normal_Flow": OperationalAdjustments(1.0,1.0,{"imaging":0.33,"lab":0.33,"bed":0.34},1.0,1.0),
        "High_Volume": OperationalAdjustments(1.1,1.1,{"imaging":0.28,"lab":0.27,"bed":0.45},0.95,1.05),
        "Resource_Constrained": OperationalAdjustments(1.15,1.05,{"imaging":0.25,"lab":0.25,"bed":0.50},0.9,1.1),
        "Critical_Surge": OperationalAdjustments(1.3,1.25,{"imaging":0.2,"lab":0.2,"bed":0.6},0.85,1.15),
        "Night_Shift": OperationalAdjustments(1.05,0.95,{"imaging":0.3,"lab":0.25,"bed":0.45},1.05,1.0),
        "Flu_Season": OperationalAdjustments(1.1,1.0,{"imaging":0.25,"lab":0.35,"bed":0.40},0.95,1.1),
    }

class EDOperationalClassifier:
    def __init__(self):
        self.operational_profiles = default_operational_profiles()
        self.pipeline = Pipeline([
            ("vect", DictVectorizer(sparse=False)),
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=(64,32), activation="relu",
                                  solver="adam", random_state=42, max_iter=500, early_stopping=True))
        ])
        self._is_fitted = False

    def fit(self, feature_dicts: List[Dict[str, Any]], labels: List[str]):
        self.pipeline.fit(feature_dicts, labels)
        self._is_fitted = True
        return self

    def predict_state(self, feature_dicts: List[Dict[str, Any]]):
        if not self._is_fitted:
            return ["Normal_Flow"] * len(feature_dicts)
        return self.pipeline.predict(feature_dicts)

    def predict_proba(self, feature_dicts: List[Dict[str, Any]]):
        if not self._is_fitted:
            import numpy as np
            return np.tile(np.array([1.0/len(ED_STATES)]*len(ED_STATES)), (len(feature_dicts),1))
        return self.pipeline.predict_proba(feature_dicts)

    def get_adjustments(self, feature_dict: Dict[str, Any]):
        probs = self.predict_proba([feature_dict])[0]
        if not self._is_fitted:
            return "Normal_Flow", probs, self.operational_profiles["Normal_Flow"]
        classes = list(self.pipeline.named_steps["mlp"].classes_)
        import numpy as np
        max_idx = int(np.argmax(probs))
        state = classes[max_idx] if classes else "Normal_Flow"
        return state, probs, self.operational_profiles[state]


In [ ]:

import datetime as dt
from typing import Any, Dict

def featurize(
    ts: dt.datetime,
    census: float,
    hist_avg: float,
    waiting_count: int,
    triage_backlog: int,
    los_p50_min: float,
    staff_to_patient_ratio: float,
    ems_runs_2h: int,
    admit_trend: float,  # -1..1
    specialist_availability: float,  # 0..1
    icu_beds_available: int,
    weather_risk: float,  # 0..1
    local_event_intensity: float,  # 0..1
    flu_index: float,  # 0..1
    equipment_downtime_flag: int,  # 0/1
    holiday_proximity_days: int,
    school_in_session: int  # 0/1
) -> Dict[str, Any]:
    hour = ts.hour
    dayofweek = ts.weekday()
    month = ts.month
    is_weekend = 1 if dayofweek >= 5 else 0
    is_night = 1 if hour < 7 or hour >= 23 else 0
    z_census = census / max(1.0, hist_avg)
    backlog_rate = triage_backlog / max(1, waiting_count if waiting_count>0 else 1)
    return {
        "hour": hour,
        "dayofweek": dayofweek,
        "month": month,
        "is_weekend": is_weekend,
        "is_night": is_night,
        "z_census": z_census,
        "waiting_count": waiting_count,
        "triage_backlog": triage_backlog,
        "backlog_rate": backlog_rate,
        "los_p50_min": los_p50_min,
        "staff_to_patient_ratio": staff_to_patient_ratio,
        "ems_runs_2h": ems_runs_2h,
        "admit_trend": admit_trend,
        "specialist_availability": specialist_availability,
        "icu_beds_available": icu_beds_available,
        "weather_risk": weather_risk,
        "local_event_intensity": local_event_intensity,
        "flu_index": flu_index,
        "equipment_downtime_flag": equipment_downtime_flag,
        "holiday_proximity_days": holiday_proximity_days,
        "school_in_session": school_in_session,
    }
